In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from tqdm import tqdm

In [2]:
# ====================== CONFIG ======================
MODEL_NAME = "rafalposwiata/deproberta-large-depression"

# Label mapping (exactly as used in the paper/repo)
LABELS = {
    0: "not depressed",
    1: "moderately depressed",
    2: "severely depressed"
}

In [3]:
# ====================== LOAD MODEL ======================
print("Loading DepRoBERTa model... (this may take a minute the first time)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Use GPU if available (Colab GPU is enabled)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Model loaded successfully on {device}!\n")

Loading DepRoBERTa model... (this may take a minute the first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: rafalposwiata/deproberta-large-depression
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully on cuda!



In [4]:
# ====================== PREDICTION FUNCTIONS ======================
def predict_text(text: str):
    """Predict depression level for a single text"""
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)[0]
        predicted_class = torch.argmax(logits, dim=1).item()
        confidence = probs[predicted_class].item()

    label = LABELS[predicted_class]
    
    print(f"Text: {text[:150]}..." if len(text) > 150 else f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)\n")
    
    return label, confidence, probs.tolist()


def predict_batch(texts: list):
    """Predict on multiple texts efficiently"""
    results = []
    for text in tqdm(texts, desc="Predicting"):
        label, confidence, probs = predict_text(text)
        results.append({
            "text": text,
            "prediction": label,
            "confidence": round(confidence, 4),
            "prob_not": round(probs[0], 4),
            "prob_moderate": round(probs[1], 4),
            "prob_severe": round(probs[2], 4)
        })
    return pd.DataFrame(results)

In [6]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

In [5]:
# ====================== EXAMPLE USAGE ======================
if __name__ == "__main__":
    test_texts = [
        "Life feels so empty lately. I don't enjoy anything anymore and just want to stay in bed.",
        "I've been feeling down for a few weeks but I'm trying to exercise and talk to people.",
        "Everything is hopeless. I can't see any future for myself. I think about ending it all."
    ]

    print("=== RUNNING EXAMPLE PREDICTIONS ===\n")
    predict_batch(test_texts)

=== RUNNING EXAMPLE PREDICTIONS ===



Predicting: 100%|██████████| 3/3 [00:00<00:00,  5.10it/s]

Text: Life feels so empty lately. I don't enjoy anything anymore and just want to stay in bed.
Prediction: moderately depressed
Confidence: 0.5881 (58.81%)

Text: I've been feeling down for a few weeks but I'm trying to exercise and talk to people.
Prediction: severely depressed
Confidence: 0.8952 (89.52%)

Text: Everything is hopeless. I can't see any future for myself. I think about ending it all.
Prediction: moderately depressed
Confidence: 0.8276 (82.76%)

